# RECELL-AI: XGBoost SOH Training Notebook
Notebook ini digunakan untuk mengolah data time-series (Voltage, Current) dari STM32 (atau NASA Dataset) untuk melatih model prediksi State of Health (SoH).

In [ ]:
!pip install pandas numpy xgboost scikit-learn matplotlib

## 1. Load Data / Generate Mock Data
Jika Anda memiliki CSV asli dari NASA atau pengujian manual, ubah bagian ini untuk membaca CSV tersebut.

In [ ]:
import pandas as pd
import numpy as np

# --- GANTI DENGAN DATA NASA ASLI JIKA ADA ---
# df = pd.read_csv('path_to_nasa_data.csv')

# Generate MOCK DATA untuk contoh
np.random.seed(42)
n_samples = 500

v_drop = np.random.uniform(0.1, 1.2, n_samples)
internal_r = np.random.uniform(0.05, 0.5, n_samples)
temp_delta = np.random.uniform(0.5, 5.0, n_samples)

soh = 100 - (v_drop * 30) - (internal_r * 50) + np.random.normal(0, 2, n_samples)
soh = np.clip(soh, 20, 100)

df = pd.DataFrame({
    'v_drop': v_drop,
    'internal_r': internal_r,
    'temp_delta': temp_delta,
    'soh': soh
})

df.head()

## 2. Train XGBoost Model
Memisahkan data menjadi Train/Test dan melatih model.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X = df[['v_drop', 'internal_r', 'temp_delta']]
y = df['soh']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Training Selesai! RMSE: {rmse:.2f}% (Rata-rata error prediksi)")

## 3. Export Model
Simpan model ke format JSON untuk digunakan di dalam program utama Jetson.

In [ ]:
model.save_model("soh_xgb_model.json")
print("Model disimpan sebagai 'soh_xgb_model.json'")

# Uncomment baris di bawah jika menjalankan di Google Colab
from google.colab import files
files.download("soh_xgb_model.json")